In [33]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

In [34]:
import langchain_core
import langchain_groq
print(langchain_core.__version__)

print(langchain_groq.__version__)

1.4.8
1.1.3


In [35]:

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [36]:
#tool create
from langchain_core.tools import tool
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    Fetches the conversion factor between a base currency and a target currency.

    Args:
        base_currency: Three-letter ISO currency code (e.g., USD)
        target_currency: Three-letter ISO currency code (e.g., INR)

    Returns:
        Conversion factor as a float.
    """

    url = f"https://open.er-api.com/v6/latest/{base_currency.upper()}"

    response = requests.get(url)
    response.raise_for_status()  # Raises an exception if the request fails

    data = response.json()

    if data["result"] != "success":
        raise ValueError("Failed to fetch exchange rate.")

    return float(data["rates"][target_currency.upper()])





In [37]:
get_conversion_factor.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR"
    }
)

96.241592

In [39]:
@tool
def currency_converter(
    base_currency: str,
    target_currency: str,
    amount: float
) -> str:
    """
    Converts an amount from one currency to another.
    """

    conversion_factor = get_conversion_factor.invoke(
        {
            "base_currency": base_currency,
            "target_currency": target_currency
        }
    )

    converted_amount = amount * conversion_factor

    return (
        f"{amount} {base_currency.upper()} = "
        f"{converted_amount:.2f} {target_currency.upper()}"
    )

In [40]:
currency_converter.invoke(
    {
        "base_currency": "USD",
        "target_currency": "INR",
        "amount": 100
    }
)

'100.0 USD = 9624.16 INR'

In [41]:
#tool binding 

llm_with_tools=llm.bind_tools([get_conversion_factor,currency_converter])


In [42]:
#tool calling

messages=HumanMessage(content="What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr")



In [43]:
messages

HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})

In [44]:
ai_message= llm_with_tools.invoke([messages])

In [46]:
ai_message.tool_calls


[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '14f76kfdw',
  'type': 'tool_call'},
 {'name': 'currency_converter',
  'args': {'amount': 10, 'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'g6d907kds',
  'type': 'tool_call'}]